# 09.01 装填因子与线性探测的访存优化章节概述

## 1. 本章前置要求

1. 编程基础：能够阅读和运行 Python、C/C++ 代码，理解数组、循环、函数、结构体和二进制文件读写。
2. 数据结构基础：理解 Hash Table（哈希表）、Hash Function（哈希函数）、key（键）与 value（值）的基本含义。
3. 算法基础：知道哈希冲突可能使多个 key 映射到同一数组槽位，并了解顺序查找的基本过程。
4. 实验环境基础：能够使用 Jupyter Notebook（交互式计算文档），并对 CANN（Compute Architecture for Neural Networks，异构计算架构）、Ascend C（昇腾 C 语言算子开发方式）和 NPU（Neural Processing Unit，神经网络处理器）有初步认识。

## 2. 本章主要内容

本章围绕开放寻址哈希表中的探测路径展开。课程先说明哈希地址如何把 key 映射到数组槽位，再介绍 Linear Probing（线性探测）遇到冲突后顺序检查后继槽位的规则。随后使用 EMPTY（空槽）、FULL（有效槽）和 Tombstone（墓碑状态）描述删除后的正确查询语义，并通过 Load Factor（装填因子）与 Average Search Length（平均查找长度，简称 ASL）分析探测链的平均成本和长尾成本。最后将只读哈希表组织为 SoA（Structure of Arrays，结构分离数组），完成批量 query 的多核 Tiling（数据切分）、Ascend C Kernel（NPU 核函数）、C++ Host（主机侧程序）和 CPU Golden（CPU 参考结果）验证。

### 2.1 哈希地址与线性探测

哈希函数首先计算初始槽位 `home=H(key)&(M-1)`。当 `home` 已被其他 key 占用时，线性探测按 `(home+step)&(M-1)` 依次检查后继槽位。表长 `M` 取 2 的幂后，按位与能够完成回绕，并保证 Python、Host 和 Kernel 使用同一条地址公式。

In [ ]:
U32_MASK = 0xFFFFFFFF

def hash32(key):
    # Python 整数不会自动发生 uint32 溢出，因此显式截断到 32 位。
    value = key & U32_MASK
    value = (value * 0x9E3779B1) & U32_MASK
    value ^= value >> 16
    return value & U32_MASK

def probe_slots(key, table_size, max_probe):
    home = hash32(key) & (table_size - 1)
    return [(home + step) & (table_size - 1) for step in range(max_probe)]

print("key=27 的前 6 个候选槽：", probe_slots(27, 16, 6))

### 2.2 状态位与 Tombstone

`EMPTY` 表示该槽从未被占用，查询遇到它即可停止；`FULL` 表示槽内保存有效 key/value；`TOMBSTONE` 表示槽中记录已被删除，但后方仍可能存在同一探测链上的 key，因此查询必须继续。独立状态数组也允许任意 `int32` key 和 value，不需要预留特殊数值充当空槽标记。

![线性探测状态与访问路径](images/linear_probe_states.png)

### 2.3 装填因子、ASL 与长尾

装填因子定义为 `α=n/M`，其中 `n` 是有效记录数，`M` 是槽位数。`α` 增大时空槽减少，线性探测更容易形成连续聚集。除平均探测长度外，还应统计 P95（第 95 百分位）、P99（第 99 百分位）和最大探测长度，因为少量长探测 query 会增加批处理尾延迟。

![装填因子与探测长度分布](images/load_factor_probe_tail.png)

### 2.4 SoA 与按需读取 value

表使用 `table_keys[M]`、`table_values[M]` 和 `states[M]` 三个连续数组。查询先读取 state：遇 `EMPTY` 停止，遇 `TOMBSTONE` 继续，只有 `FULL` 才比较 key；确认命中后才读取 value。该顺序避免未命中查询搬运无用 value，也使输入输出接口保持清晰。

### 2.5 NPU 映射、Tiling 与 UB 预算

Host 负责构建只读表、检查装填因子与最大探测距离，并把数组传入 Device（设备侧）。Kernel 按连续 query 区间分核，以 128 个 `int32` query 为一个 Tile（分块）搬入 UB（Unified Buffer，统一缓冲区）。三个输出也在 UB 中连续生成后写回；表数组保留在 GM（Global Memory，全局内存）中，由运行时槽位下标访问。

![开放寻址批量查询的 Host 与 NPU 数据流](images/open_addressing_npu_dataflow.png)

### 2.6 复杂度与实验边界

在装填因子受控且哈希分布合理时，开放寻址查询的期望复杂度接近 `O(1)`；最坏情况下可能检查 `MAX_PROBE` 个槽。本实验只处理 Host 已构建完成的只读表，Kernel 不执行插入、删除和扩容。`MAX_PROBE` 是明确的执行上界，Host 必须保证有效 key 的实际探测距离不超过该值。

### 2.7 课后练习

1. 为什么查询遇到 `EMPTY` 可以停止，而遇到 `TOMBSTONE` 必须继续？
2. 表长 `M=256`、有效记录数 `n=128` 时，装填因子是多少？
3. 为什么输出必须同时包含 `out_values` 和 `hit_flags`？
4. `Q=1000`、可用 Vector Core（向量计算核心）数为 20 时，`blockNum` 和 `queriesPerCore` 分别是多少？

In [ ]:
!cat answer/09.01_chapter_intro/answers.md

## 3. 学习目标

完成本章后，学习者能够：

1. 写出哈希初始槽位和线性探测序列的计算公式。
2. 解释 EMPTY、FULL、TOMBSTONE 对查询停止条件的影响。
3. 计算装填因子，并使用平均、P99 和最大探测长度描述访存成本。
4. 说明 SoA 布局、按需读取 value 和连续 query Tiling 的作用。
5. 使用 CPU Golden 验证 NPU 的 value、hit 与 probe count 输出。

## 4. 小节简介与跳转链接

<table align="left" style="width: 80%; max-width: 1200px; margin: 0 auto 0 0 !important; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">小节</th>
      <th style="text-align: left;">简介</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><a href="./09.01_chapter_intro.ipynb" target="_self">09.01 装填因子与线性探测的访存优化章节概述</a></td>
      <td style="text-align: left;">介绍前置要求、线性探测、状态位、装填因子、SoA、NPU 映射和学习目标。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><a href="./09.02_open_addressing_lookup.ipynb" target="_self">09.02 开放寻址批量查询实验</a></td>
      <td style="text-align: left;">完成 CPU 构表、探测统计、Tiling、Ascend C Kernel、C++ Host、构建运行和结果验证。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><a href="./09.03_chapter_practice.ipynb" target="_self">09.03 章节实践</a></td>
      <td style="text-align: left;">补全线性探测、墓碑语义、探测统计和多核 Tiling。</td>
    </tr>
  </tbody>
</table>
<div style="clear: both;"></div>